# DEKM 2021 release-faithful reproduction: REUTERS seeds 42-44

This notebook runs the released 2021 algorithmic protocol on the current Kaggle GPU runtime. It records the modern runtime explicitly; this is not a bitwise TensorFlow 2.4.1 environment reproduction.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a GPU accelerator in Kaggle notebook settings."

In [ ]:
repo_url = "https://github.com/vdat-18/mvdec_fpmax.git"
repo_dir = "/kaggle/working/mvdec_fpmax"
!rm -rf {repo_dir}
!git clone --branch dev --single-branch {repo_url} {repo_dir}
%cd {repo_dir}
!python -m pip install -q -e . gdown

In [ ]:
!python scripts/fetch_dekm_release.py --datasets REUTERS

output_dir = '/kaggle/working/dekm-release-2021'
!dekm-reproduce REUTERS \
    --protocol release-2021 \
    --seed 42 --runs 3 --require-gpu \
    --dataset-root external_repos/DEKM/datasets \
    --output-dir {output_dir}


In [ ]:
import json
from pathlib import Path
import pandas as pd

root = Path(output_dir) / 'reuters' / 'dekm_2021_release_seeded_modern_runtime_v1'
config_root = next(path for path in root.iterdir() if path.is_dir())
summary = pd.read_csv(config_root / 'dekm_summary.csv')
runs = pd.read_csv(config_root / 'dekm_runs.csv')
manifest = json.loads((config_root / 'seed_42' / 'manifest.json').read_text())
assert manifest['runtime']['tensorflow_op_determinism'] is False
assert manifest['protocol_id'] == 'dekm_2021_release_seeded_modern_runtime_v1'
display(runs[['seed', 'acc', 'nmi', 'stop_reason', 'fit_time_seconds']])
display(summary)
print(json.dumps(manifest['runtime'], indent=2))
print('Dataset SHA-256:', manifest['source_sha256'])
print('Config SHA-256:', manifest['config_hash'])


In [ ]:
!cd /kaggle/working && zip -qr dekm-reuters-release-2021-seeds42-44.zip dekm-release-2021
print('Result: /kaggle/working/dekm-reuters-release-2021-seeds42-44.zip')
